<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
تشخیص جنسیت
</font>
</h1>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مقدمه و صورت مسئله
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    امروزه شبکه‌های اجتماعی کاربردهای گسترده‌ای دارند. اولین استفاده آن، تفریح و اوقات فراغت است. اما از دید دیگر، می‌توان از شبکه‌های اجتماعی برای پیدا کردن الگوهای رفتاری استفاده کرد. به عنوان مثال با تحلیل نظرات کاربران شبکه‌های اجتماعی می‌توانیم ضعف‌های کسب و کار خود را پیدا کنیم.
    <br>
    جنسیت یکی از پارامترهای تاثیرگذار در رفتار کاربران است. در مواجه با یک موضوع، بانوان عمدتا یک‌طور واکنش نشان خواهند داد و آقایان نیز طور دیگری!
    <br>
    حال در این تمرین قصد داریم با اطلاعاتی که دیتاک از کاربران توییتر و اینستاگرم در اختیار ما قرار داده است، جنسیت آن‌ها را پیش‌بینی کنیم.
</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ابتدا کتابخانه‌های مورد نیازتان را وارد کنید.
</font>
</p>

In [ ]:
import numpy as np
import pandas as pd
import emoji
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
     !pip install emoji

In [ ]:
import emoji

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
مجموعه داده آموزش ۸۰۰۰ سطر و ۱۰ ستون دارد. در جدول زیر، اطلاعات بیشتری در مورد داده نوشته شده است.     
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|gender|جنسیت (ستون هدف)|
|age|بازه سنی کاربر|
|fullname|نامی که در پروفایل شبکه اجتماعی نوشته شده است |
|username|نام کاربری|
|biography|بیوگرافی شبکه اجتماعی کاربر|
|follower_count|تعداد دنبال‌کننده‌های کاربر|
|following_count|تعداد افرادی که کاربر را دنبال می‌کنند|
|is_business|کسب‌وکاری بودن حساب کاربری|
|is_verified|تایید شده بودن حساب کاربری|
|is_private|خصوصی بودن حساب کاربری|
    
</font>
</div>
</center>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    ستون <code>age</code> متغیر پیوسته نیست؛ بلکه دسته‌های سنی را نشان می‌دهد. در جدول زیر نحوه نگاشت سن را مشاهده می‌کنیم.
</font>
</p>


<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|سن واقعی کاربران|عدد نگاشت شده|
|:------:|:---:|
|1|کمتر از ۱۸ سال|
|2|از ۱۹ تا ۲۹ سال|
|3|از ۳۰ تا ۴۰ سال|
|4|بالاتر از ۴۰ سال|
    
</font>
</div>
</center>


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در ابتدا نیاز است فایل‌های مجموعه‌داده را بخوانید. نمونه‌های آموزشی در فایل <code>train_data.csv</code> و نمونه‌های آزمون که باید دسته‌ی آن‌ها را پیش‌بینی کنید در فایل <code>test_data.csv</code> ذخیره شده‌اند. اگر لازم دانستید می‌توانید به دلخواه خود بخشی از دادگان آموزشی را به عنوان دادگان اعتبارسنجی نیز جدا کنید.
</font>
</p>

In [ ]:
train = pd.read_csv('/content/drive/My Drive/ML/gender_detection_2/data/train_data.csv')
test = pd.read_csv('/content/drive/My Drive/ML/gender_detection_2/data/test_data.csv')


In [ ]:
train

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
پیش‌پردازش و مهندسی ویژگی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    در این سوال شما می‌توانید از هر تکنیک پیش‌پردازش/مهندسی ویژگی که در فصل‌های گذشته آموختید، استفاده کنید.
    <br>
    تکنیک‌هایی که استفاده می‌کنید به شکل مستقیم مورد ارزیابی توسط سامانه داوری قرار <b>نمی‌گیرند.</b> بلکه همه آن‌ها در دقت مدل شما تاثیر خواهند گذاشت؛ بنابراین هر چه پیش‌پردازش/مهندسی ویژگی بهتری انجام دهید تا دقت مدل بهبود پیدا کند، امتیاز بیشتری از این سوال کسب خواهید کرد.
</font>
</p>

In [ ]:
train['gender'] = train['gender'].map({'man': 0, 'woman': 1})

In [ ]:
print(train.isna().sum())
train.dtypes
print(test.isna().sum())
test.dtypes

In [ ]:
VehicleClass_mode = train['is_business'].mode()[0]
train['is_business'].fillna(value=VehicleClass_mode, inplace=True)

In [ ]:
def extract_emojis(text):
    if pd.isna(text):
        return []
    return [c for c in text if c in emoji.distinct_emoji_list(text)]


def create_features(df):

    df['fullname'] = df['fullname'].fillna('').str.lower()
    df['username'] = df['username'].fillna('').str.lower()
    df['biography'] = df['biography'].fillna('').str.lower()


    df['fullname_len'] = df['fullname'].apply(len)
    df['username_len'] = df['username'].apply(len)
    df['biography_len'] = df['biography'].apply(len)


    df['emoji_count'] = df['biography'].apply(lambda x: len(extract_emojis(x)))


    df['follower_following_ratio'] = np.where(
        df['following_count'] > 0,
        df['follower_count'] / df['following_count'],
        0
    )


    df['is_business'] = df['is_business'].fillna(0).astype(int)
    df['is_verified'] = df['is_verified'].fillna(0).astype(int)
    df['is_private'] = df['is_verified'].fillna(0).astype(int)

    return df


train = create_features(train)
test = create_features(test)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
مدل‌سازی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    حال که داده را پاکسازی کرده و احتمالا ویژگی‌هایی را به آن افزوده یا از آن حذف کرده‌اید، وقت آن است که مدلی آموزش دهید که بتواند متغیر هدف این مسئله را پیش‌بینی کند.
</font>
</p>

In [ ]:
# ترکیب ستون‌های متنی
train['combined_text'] = train['fullname'] + " " + train['username'] + " " + train['biography']
test['combined_text'] = test['fullname'] + " " + test['username'] + " " + test['biography']


X = train.drop('gender', axis=1)
y = train['gender']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


text_vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words='english')
X_train_text = text_vectorizer.fit_transform(X_train['combined_text'])
X_val_text = text_vectorizer.transform(X_val['combined_text'])


text_model = XGBClassifier(random_state=42, eval_metric='logloss', n_estimators=150, max_depth=7)
text_model.fit(X_train_text, y_train)

X_train['text_predicted_gender'] = text_model.predict(X_train_text)
X_val['text_predicted_gender'] = text_model.predict(X_val_text)

In [ ]:

numeric_features = ['age', 'follower_count', 'following_count', 'fullname_len',
                   'username_len', 'biography_len', 'emoji_count',
                   'follower_following_ratio', 'text_predicted_gender']
boolean_features = ['is_business', 'is_verified', 'is_private']

model = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('bool', 'passthrough', boolean_features)
    ])),
    ('classifier', GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42))
])


model.fit(X_train[numeric_features + boolean_features], y_train)

<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
معیار ارزیابی
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>f1_score</code> نام دارد.
    <br>
    این معیار، سنجه ارزیابی کیفیت مدل شماست. به عبارت بهتر در سامانه داوری هم از همین معیار برای نمره‌دهی استفاده شده است.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی دادگان آموزش یا اعتبارسنجی ارزیابی کنید.
</font>
</p>

In [ ]:

y_pred =model.predict(X_val[numeric_features + boolean_features])
print(classification_report(y_val, y_pred))
print(f"F1 Score: {f1_score(y_val, y_pred):.4f}")

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font color="red"><b color='red'>توجه:</b></font>
<font face="vazirmatn" size=3>
 جهت کسب امتیاز کامل نیاز است تا پاسخ شما حداقل امتیاز <code>75%</code> را با توجه به معیار معرفی‌شده کسب نماید.
</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    پس از مهندسی ویژگی و مدلسازی، الگوریتمی دارید که می‌تواند شما را از متغیرهای مستقل به متغیر هدف برساند.
    <br>
    از این مدل برای پیش‌بینی نمونه‌های موجود در داده تست استفاده کنید و نتایج را در قالب جدول (<code>dataframe</code>) زیر آماده کنید.
</font>
</p>

<div dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
|ستون|توضیحات|
|:------:|:---:|
|gender|جنسیت کاربر|
    
</font>
</div>



<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    اسم دیتافریم باید <i>submission</i> باشد؛ در غیر این صورت، سامانه داوری نمی‌تواند تلاش‌ شما را ارزیابی کند.
    <br>
    این دیتافریم تنها شامل ۱ ستون با اسم <i>gender</i> است و ۲۰۰۰ سطر دارد.
    <br>
    به ازای هر سطر موجود در دیتافریم <i>test_data</i> شما باید یک مقدار پیشبینی شده داشته باشید.
    <br>
    جدول زیر، ۵ سطر ابتدایی دیتافریم <code>submission</code> را نشان می‌دهد. البته در جواب شما، مقادیر ستون <i>gender</i> ممکن است متفاوت باشد.
</font>
</p>

<div style="text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazirmatn" size=3>
    
||gender|
|----|-----|
|0|woman|
|1|woman|
|2|man|
|3|woman|
|4|man|

</font>
</div>



In [ ]:

test['combined_text'] = test['fullname'] + " " + test['username'] + " " + test['biography']
X_test_text = text_vectorizer.transform(test['combined_text'])
test['text_predicted_gender'] = text_model.predict(X_test_text)

test_predictions = model.predict(test[numeric_features + boolean_features])


submission = pd.DataFrame({
    'gender': ['woman' if pred == 1 else 'man' for pred in test_predictions]
})


submission.to_csv('submission.csv', index=False)

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazirmatn" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazirmatn" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) تا در صورت نیاز به پشتیبانی امکان بررسی کد شما وجود داشته باشد.
</font>
</p>

In [ ]:
import zipfile
import joblib
import os
output_path = "/content/drive/My Drive/ML/gender_detection_2"
os.makedirs(output_path, exist_ok=True)
def compress(file_names):
    compression = zipfile.ZIP_DEFLATED
    zip_file_path = os.path.join(output_path, "result.zip")
    with zipfile.ZipFile(zip_file_path, mode="w") as zf:
        for file_name in file_names:
            if os.path.exists(file_name):
                zf.write(file_name, os.path.basename(file_name), compress_type=compression)
            else:
                print(f"⚠️ Warning: {file_name} does not exist and will be skipped.")

submission.to_csv('submission.csv', index=False)
file_names = ['gender_detection.ipynb', 'submission.csv']
compress(file_names)